# AbAffinity — custom dataset: zero-shot + few-shot, Integrated Gradients & structure mapping

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshitsinghsnu/AgAbAffinity/blob/main/notebooks/AgAbGated_Custom_ZeroShot_FewShot_IG.ipynb)

Run the **final SAaIntDB All-CDR model** (random-split best fold) on **your own antibody-antigen
dataset**, end to end:

1. Load the shipped checkpoint (`model_weights/saaintdb_allcdr_random_bestfold.pt`) and its config.
2. Embed your sequences with frozen **ESM-2 650M** (heavy = CDR-H1/H2/H3 pooled; light & antigen = mean).
3. **Zero-shot** affinity prediction (no fine-tuning) -> Pearson / Spearman / RMSE.
4. **Few-shot** fine-tuning on a fraction of your labels (validation early-stopping, held-out test).
5. **Integrated Gradients** per-residue attribution -> heatmap.
6. Map high-attribution residues onto the **3D structure**, rendered inline.

### Input format
A CSV with four columns: **`light`, `heavy`, `antigen`, `Y`** (one row per complex; `Y` = affinity,
e.g. pKd). Nanobodies: put the heavy sequence in the `light` column too.

> **Google Colab:** click the badge above, then **Runtime -> Run all**. The first code cell clones
> the repo and installs `transformers`, `captum` and `py3Dmol`. **Running locally:** the setup cell
> is a no-op; just make sure the `AbAffinity` package is importable (`pip install -e .`). A GPU is
> recommended for ESM-2 embedding but not required.

In [1]:
# === Google Colab setup — run me first (no-op when running locally) ===
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/harshitsinghsnu/AgAbAffinity.git'   # <- public repo URL
    REPO_DIR = '/content/AgAbAffinity'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, 'notebooks'))   # next cell finds the repo from here
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'transformers', 'captum', 'py3Dmol'], check=True)   # torch ships with Colab
    print('Colab ready - repo at', REPO_DIR)
else:
    print('Local run - using the working-copy repo.')

Colab ready - repo at /content/AgAbAffinity


## 1 - Setup, config and checkpoint

In [2]:
import os, sys, json, re, warnings
from pathlib import Path
import numpy as np, pandas as pd, torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
try:
    import yaml
except Exception:
    yaml = None
warnings.filterwarnings('ignore')

# Locate repo root whether launched from the repo root or from notebooks/
CWD  = Path.cwd()
REPO = CWD.parent if CWD.name == 'notebooks' else CWD
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from AbAffinity.models.mutual_strong import MutualTriStreamStrong

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
CKPT_PATH = REPO / 'model_weights' / 'saaintdb_allcdr_random_bestfold.pt'
CFG_YAML  = REPO / 'configs' / 'saaintdb' / 'sa_ours_allcdr_random.yaml'

print('Repo root      :', REPO)
print('Device         :', DEVICE)
print('Checkpoint     :', CKPT_PATH, '| exists:', CKPT_PATH.exists())
if yaml and CFG_YAML.exists():
    print('Experiment cfg :', yaml.safe_load(open(CFG_YAML)))   # pooling=allcdr, split=random


Repo root      : /content/AgAbAffinity
Device         : cpu
Checkpoint     : /content/AgAbAffinity/model_weights/saaintdb_allcdr_random_bestfold.pt | exists: True
Experiment cfg : {'name': 'sa_ours_allcdr_random', 'pooling': 'allcdr', 'split': 'random', 'n_folds': 10}


In [3]:
# Load the checkpoint (stores model_state_dict, config, pkd_bounds, val_metrics)
ckpt       = torch.load(CKPT_PATH, map_location=DEVICE)
CFG        = ckpt['config']
PKD_BOUNDS = tuple(ckpt['pkd_bounds'])          # SAaIntDB training-fold affinity bounds
print('Model config        :', CFG)
print('SAaIntDB pKd bounds :', PKD_BOUNDS)
print('Best-fold val metrics:', ckpt.get('val_metrics'))

def build_model(cfg, device=DEVICE):
    return MutualTriStreamStrong(
        esm_dim=1280,
        projected_size=cfg.get('projected_size', 256),
        num_heads=cfg.get('num_heads', 8),
        dropout=0.0,
        n_layers=cfg.get('n_layers', 2),
        device=device,
    ).to(device)

model = build_model(CFG)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print('\nLoaded MutualTriStreamStrong (All-CDR, SAaIntDB random best fold).')


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy.core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy.core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn mor

## 2 - Frozen ESM-2 650M embedder

Heavy chain is pooled over its **CDR-H1/H2/H3** (IMGT via ANARCI, regex fallback if ANARCI/HMMER is
unavailable); light chain and antigen are **mean-pooled** - exactly the final-model convention.

In [ ]:
from transformers import EsmTokenizer, EsmModel
print('Loading ESM-2 650M (facebook/esm2_t33_650M_UR50D) ...')
TOK = EsmTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
ESM = EsmModel.from_pretrained('facebook/esm2_t33_650M_UR50D').to(DEVICE).eval()

IMGT_CDR = {*range(27, 39), *range(56, 66), *range(105, 118)}   # CDR1 / CDR2 / CDR3 (IMGT)

def cdr_idx(seq):
    """0-based indices of heavy-chain CDR residues (IMGT via ANARCI; regex fallback)."""
    try:
        import anarci
        r = anarci.anarci([('q', seq)], scheme='imgt', assign_germline=False,
                          allow=set('ACDEFGHIKLMNPQRSTVWY'))
        if r and r[0] and r[0][0] and r[0][0][0] is not None:
            idx, sp = [], 0
            for (p, _), aa in r[0][0][0]:
                if aa == '-':
                    continue
                if p in IMGT_CDR:
                    idx.append(sp)
                sp += 1
            if idx:
                return sorted(set(idx))
    except Exception:
        pass
    o = []
    m = re.search(r'C[A-Z]{2,5}[SAGTV]([A-Z]{8,15})WVRQ', seq); o += list(range(*m.span(1))) if m else list(range(26, 36))
    m = re.search(r'W[VI]RQ[A-Z]{6,14}W[VL][AS]([A-Z]{10,20})VKGRF', seq); o += list(range(*m.span(1))) if m else list(range(50, 64))
    m = re.search(r'WYYCA([A-Z]+)WGQGT', seq) or re.search(r'WYYC[A-Z]([A-Z]+)WGQG', seq); o += list(range(*m.span(1))) if m else list(range(95, 110))
    return sorted(set(o))

@torch.no_grad()
def esm_residues(seq):
    """Per-residue ESM-2 embeddings (L, 1280), CLS/EOS stripped."""
    inp = TOK([seq], return_tensors='pt', truncation=True, max_length=1024).to(DEVICE)
    te  = ESM(**inp).last_hidden_state[0]
    L   = int(inp['attention_mask'][0].sum()) - 2
    return te[1:L + 1]

def pool_heavy_cdr(seq):
    res = esm_residues(seq); L = res.shape[0]
    ix  = [p for p in cdr_idx(seq) if 0 <= p < L]
    return (res[ix].mean(0) if ix else res.mean(0)).cpu().numpy().astype(np.float32)

def pool_mean(seq):
    return esm_residues(seq).mean(0).cpu().numpy().astype(np.float32)

def embed_col(seqs, fn):
    cache, out = {}, []
    for s in seqs:
        if s not in cache:
            cache[s] = fn(s)
        out.append(cache[s])
    return np.stack(out)


## 3 - Load your dataset

Set `DATA_CSV` to a CSV with columns `light, heavy, antigen, Y`. In Colab you can instead use the
upload helper. (If the file is missing, the zero/few-shot sections are skipped, but the IG and
structure demos in sections 6-7 still run on the bundled example complexes.)

In [ ]:
# Option A - Colab upload:
# from google.colab import files; up = files.upload(); DATA_CSV = Path(list(up)[0])
# Option B - point to a CSV on disk:
DATA_CSV = REPO / 'data' / 'sample_saaintdb_subset.csv'   # demo: 60-complex SAaIntDB subset (replace with your CSV)

df = None
if Path(DATA_CSV).exists():
    df = pd.read_csv(DATA_CSV)
    df.columns = [c.lower().strip() for c in df.columns]
    need = {'light', 'heavy', 'antigen', 'y'}
    assert need.issubset(df.columns), f'CSV must have columns {need}; got {list(df.columns)}'
    df = df[['light', 'heavy', 'antigen', 'y']].dropna().reset_index(drop=True)
    df['y'] = df['y'].astype(float)
    print(f'Loaded {len(df)} complexes | Y range [{df.y.min():.2f}, {df.y.max():.2f}]')
    display(df.head())
else:
    print(f'No dataset at {DATA_CSV}.')
    print('Set DATA_CSV to a CSV with columns: light, heavy, antigen, Y  '
          '(sections 4-5 need it; 6-7 run without it).')

In [ ]:
# Embed the dataset (heavy = CDR-pool, light/antigen = mean). Unique sequences are cached.
H = L = A = Y = None
if df is not None:
    print('Embedding with ESM-2 ...')
    H = embed_col(df.heavy,   pool_heavy_cdr)
    L = embed_col(df.light,   pool_mean)
    A = embed_col(df.antigen, pool_mean)
    Y = df.y.values.astype(np.float32)
    print('Embedded:', H.shape, L.shape, A.shape)


## 4 - Zero-shot prediction (no fine-tuning)

The frozen model maps each complex to a cosine score, converted to the pKd scale with the model's
**SAaIntDB** training bounds. Pearson and Spearman are scale-invariant; RMSE assumes your `Y` is on
a comparable pKd scale.

In [ ]:
@torch.no_grad()
def predict(m, Hh, Ll, Aa, bounds, bs=256):
    lo, hi = bounds; out = []
    for i in range(0, len(Hh), bs):
        h = torch.tensor(Hh[i:i+bs]).to(DEVICE)
        l = torch.tensor(Ll[i:i+bs]).to(DEVICE)
        a = torch.tensor(Aa[i:i+bs]).to(DEVICE)
        cos = m(l, h, a)['cosine_similarity'].cpu().numpy()
        out.extend(((cos + 1) / 2 * (hi - lo) + lo).tolist())
    return np.array(out)

def metrics(t, p):
    return dict(pearson=float(pearsonr(t, p)[0]),
                spearman=float(spearmanr(t, p)[0]),
                rmse=float(np.sqrt(np.mean((t - p) ** 2))))

if df is not None:
    pred_zs = predict(model, H, L, A, PKD_BOUNDS)
    m_zs = metrics(Y, pred_zs)
    print('ZERO-SHOT:', {k: round(v, 4) for k, v in m_zs.items()})
    plt.figure(figsize=(4.6, 4.6))
    plt.scatter(Y, pred_zs, s=10, alpha=0.5, edgecolor='none')
    plt.xlabel('Measured Y'); plt.ylabel('Predicted (pKd scale)')
    plt.title(f"Zero-shot   r={m_zs['pearson']:.3f}   rho={m_zs['spearman']:.3f}")
    plt.tight_layout(); plt.show()


## 5 - Few-shot fine-tuning

Fine-tune the SAaIntDB model on a fraction of your labels. We hold out a small **validation** split
from the training fraction for early stopping (Pearson) and report on a disjoint **test** split -
no test-set selection, calibration bounds taken from the training split only.

In [ ]:
FRAC, SEED, EPOCHS, LR, WD, BS = 0.30, 42, 40, 5e-5, 0.01, 16
from AbAffinity.utils.main_symmetric_mean import setup_reproducibility

if df is not None and len(df) >= 10:
    setup_reproducibility(SEED)
    n   = len(df)
    idx = np.random.RandomState(SEED).permutation(n)
    ntr = max(5, int(n * FRAC))
    tr, te = idx[:ntr], idx[ntr:]
    nval   = max(2, int(0.15 * len(tr)))
    val, trn = tr[:nval], tr[nval:]
    lo, hi = float(Y[trn].min()), float(Y[trn].max())   # calibrate on TRAIN only

    ft  = build_model(CFG); ft.load_state_dict(ckpt['model_state_dict'])
    opt = torch.optim.AdamW(ft.parameters(), lr=LR, weight_decay=WD)
    def get(ix):
        return (torch.tensor(L[ix]).to(DEVICE), torch.tensor(H[ix]).to(DEVICE),
                torch.tensor(A[ix]).to(DEVICE), torch.tensor(Y[ix]).to(DEVICE))

    best, best_state = -9.0, None
    for ep in range(EPOCHS):
        ft.train(); perm = np.random.permutation(trn)
        for i in range(0, len(perm), BS):
            bi = perm[i:i+BS]; l, h, a, y = get(bi)
            tgt = 2 * (y - lo) / (hi - lo) - 1
            loss = F.mse_loss(ft(l, h, a)['cosine_similarity'], tgt)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(ft.parameters(), 1.0); opt.step()
        pv = predict(ft, H[val], L[val], A[val], (lo, hi))
        rv = float(pearsonr(Y[val], pv)[0])
        if rv > best:
            best, best_state = rv, {k: v.cpu().clone() for k, v in ft.state_dict().items()}
    ft.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    pred_fs = predict(ft, H[te], L[te], A[te], (lo, hi))
    m_fs = metrics(Y[te], pred_fs)
    print(f'FEW-SHOT ({int(FRAC*100)}% fine-tune, eval on held-out n={len(te)}):',
          {k: round(v, 4) for k, v in m_fs.items()})
    if df is not None and 'm_zs' in dir():
        print('Zero-shot -> few-shot Pearson: '
              f"{m_zs['pearson']:.3f} -> {m_fs['pearson']:.3f}")
else:
    print('Few-shot needs a dataset with >= 10 rows; skipped.')


## 6 - Integrated Gradients per-residue attribution

Wrap the model in a differentiable mean-pool layer so Captum IG can attribute the predicted pKd back
to individual residue embeddings. By default we attribute one of the bundled structure-backed
complexes (so section 7 can map onto its PDB); set `EXAMPLE_ROW` to an integer to attribute one of
your own complexes instead.

In [ ]:
from captum.attr import IntegratedGradients

class PerResidue(torch.nn.Module):
    def __init__(self, m, bounds):
        super().__init__(); self.m = m; self.lo, self.hi = bounds
    def forward(self, le, he, ae):
        out = self.m(le.mean(1), he.mean(1), ae.mean(1))
        cos = out['cosine_similarity']
        return ((cos + 1) / 2 * (self.hi - self.lo) + self.lo).squeeze()

def run_ig(m, bounds, light, heavy, antigen, steps=50):
    def t(seq):
        x = esm_residues(seq).detach().cpu().numpy()[None]
        return torch.tensor(x, dtype=torch.float32, requires_grad=True).to(DEVICE)
    le, he, ae = t(light), t(heavy), t(antigen)
    ig  = IntegratedGradients(PerResidue(m, bounds).to(DEVICE).eval())
    att = ig.attribute((le, he, ae),
                       baselines=(torch.zeros_like(le), torch.zeros_like(he), torch.zeros_like(ae)),
                       n_steps=steps)
    return [a.squeeze(0).sum(-1).detach().cpu().numpy() for a in att]   # light, heavy, antigen

def heatmap(attrs, seqs, names, title, cols=20):
    cmaps = ['Blues', 'Greens', 'Purples']
    fig, axes = plt.subplots(len(attrs), 1, figsize=(max(12, cols*0.55), len(attrs)*2.8))
    if len(attrs) == 1: axes = [axes]
    for ax, attr, seq, nm, cm in zip(axes, attrs, seqs, names, cmaps):
        a = (attr - attr.min()) / (attr.max() - attr.min() + 1e-8)
        Ln = len(a); rows = int(np.ceil(Ln/cols)); pad = rows*cols - Ln
        grid = np.pad(a, (0, pad), constant_values=np.nan).reshape(rows, cols)
        sg   = np.array(list(seq) + ['']*pad).reshape(rows, cols)
        ax.imshow(grid, cmap=cm, vmin=0, vmax=1, aspect='auto')
        for i in range(rows):
            for j in range(cols):
                if sg[i, j]:
                    ax.text(j, i, sg[i, j], ha='center', va='center', fontsize=6,
                            color='white' if grid[i, j] > 0.6 else 'black')
        ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f'Residue importance: {nm}', fontsize=10)
    fig.suptitle(title, fontsize=12, fontweight='bold'); plt.tight_layout(); plt.show()

COMPLEXES = json.load(open(REPO / 'data' / 'complexes.json'))
EXAMPLE_ROW = None          # set to an int to use df.iloc[EXAMPLE_ROW]; else a bundled complex
EXAMPLE_PDB = '1VFB'        # bundled complex used when EXAMPLE_ROW is None (must have a structure)

if EXAMPLE_ROW is not None and df is not None:
    ex = dict(light=df.light[EXAMPLE_ROW], heavy=df.heavy[EXAMPLE_ROW], antigen=df.antigen[EXAMPLE_ROW])
    ex_name = f'custom row {EXAMPLE_ROW}'
else:
    ex = COMPLEXES[EXAMPLE_PDB]; ex_name = EXAMPLE_PDB

la, ha, aa = run_ig(model, PKD_BOUNDS, ex['light'], ex['heavy'], ex['antigen'])
heatmap([la, ha, aa], [ex['light'], ex['heavy'], ex['antigen']],
        ['Light', 'Heavy', 'Antigen'], f'Integrated Gradients - {ex_name}')


## 7 - Map attribution onto the 3D structure (paper-style composite)

Matching the paper's interface composite: a central **overview** of the complex (Heavy = green,
Light = blue, Antigen = purple) with **3D arrows** pointing at each chain's high-attribution interface
region, plus **three per-chain inset panels** showing that chain's residues as sticks + Cα spheres
with **RESN-RESI labels**. Colour runs **light to dark within each chain's colormap - darkest =
highest attribution**. Residues with normalised attribution > `ATTR_THRESHOLD` are drawn; the
`TOP_LABELS` highest per chain are labelled. Inline `py3Dmol` (positional sequence->residue mapping;
set `CHAIN_MAP` for your PDB).

In [ ]:
# pip install py3Dmol   # if needed
import urllib.request
import matplotlib.cm as cm
try:
    import py3Dmol
except Exception:
    print('py3Dmol not installed - run:  pip install py3Dmol'); py3Dmol = None

PDB_ID         = EXAMPLE_PDB     # complex IG was run on above
ATTR_THRESHOLD = 0.4             # residues above this (normalised) are drawn as sticks
TOP_LABELS     = 12              # label this many highest-attribution residues per chain

CHAIN_MAP = {
    '1VFB': {'heavy': 'B', 'light': 'A', 'antigen': 'C'},
    '4ETQ': {'heavy': 'H', 'light': 'L', 'antigen': 'C'},
    '5GRJ': {'heavy': 'H', 'light': 'L', 'antigen': 'A'},
    '5Y9J': {'heavy': 'H', 'light': 'L', 'antigen': 'A'},
}
ROLE = {                                            # heatmap colours: H=Greens, L=Blues, Ag=Purples
    'heavy':   {'attr': ha, 'cmap': cm.get_cmap('Greens'),  'base': 0.18, 'arrow': 'green'},
    'light':   {'attr': la, 'cmap': cm.get_cmap('Blues'),   'base': 0.16, 'arrow': 'blue'},
    'antigen': {'attr': aa, 'cmap': cm.get_cmap('Purples'), 'base': 0.22, 'arrow': 'purple'},
}
def hexcol(cmap, t):
    c = cmap(float(np.clip(t, 0, 1)))
    return '0x{:02X}{:02X}{:02X}'.format(int(c[0]*255), int(c[1]*255), int(c[2]*255))

if py3Dmol is not None and PDB_ID in CHAIN_MAP:
    chains = CHAIN_MAP[PDB_ID]
    pdb = urllib.request.urlopen(f'https://files.rcsb.org/download/{PDB_ID}.pdb').read().decode()
    # ordered (resi, resn) and CA xyz per chain
    chain_res, chain_xyz = {}, {}
    for ln in pdb.splitlines():
        if ln.startswith('ATOM') and ln[12:16].strip() == 'CA':
            ch = ln[21]; resn = ln[17:20].strip(); resi = ln[22:27].strip()
            try:
                xyz = (float(ln[30:38]), float(ln[38:46]), float(ln[46:54]))
            except ValueError:
                continue
            chain_res.setdefault(ch, [])
            if not chain_res[ch] or chain_res[ch][-1][0] != resi:
                chain_res[ch].append((resi, resn))
            chain_xyz[(ch, resi)] = xyz

    def selected(role):
        ch = chains[role]; a = ROLE[role]['attr']
        hn = (a - a.min()) / (a.max() - a.min() + 1e-8)
        rr = chain_res.get(ch, []); m = min(len(hn), len(rr))
        return ch, [(float(hn[i]), rr[i][0], rr[i][1]) for i in range(m) if hn[i] > ATTR_THRESHOLD]

    def draw(role, gr, gc, labels):
        ch, sel = selected(role)
        if not sel: return
        cmap = ROLE[role]['cmap']
        vmin = min(s[0] for s in sel); vmax = max(s[0] for s in sel)
        for v, resi, resn in sel:
            t = 0.35 + 0.65 * ((v - vmin) / (vmax - vmin + 1e-8))    # light -> dark, darkest = highest
            col = hexcol(cmap, t)
            view.addStyle({'chain': ch, 'resi': resi}, {'stick': {'color': col, 'radius': 0.3}}, viewer=(gr, gc))
            view.addStyle({'chain': ch, 'resi': resi, 'atom': 'CA'}, {'sphere': {'color': col, 'scale': 0.3}}, viewer=(gr, gc))
        if labels:
            for v, resi, resn in sorted(sel, key=lambda s: -s[0])[:TOP_LABELS]:
                view.addLabel(f'{resn}{resi}',
                              {'fontSize': 10, 'fontColor': 'black', 'backgroundColor': 'white',
                               'backgroundOpacity': 0.6, 'borderThickness': 0.5},
                              {'chain': ch, 'resi': resi, 'atom': 'CA'}, viewer=(gr, gc))

    view = py3Dmol.view(width=940, height=820, viewergrid=(2, 2))
    cells = {'overview': (0, 0), 'heavy': (0, 1), 'light': (1, 0), 'antigen': (1, 1)}
    for key, (gr, gc) in cells.items():
        view.addModel(pdb, 'pdb', viewer=(gr, gc))
        for role, ch in chains.items():
            view.setStyle({'chain': ch},
                          {'cartoon': {'color': hexcol(ROLE[role]['cmap'], ROLE[role]['base']),
                                       'opacity': 0.6 if key == 'overview' else 0.45}}, viewer=(gr, gc))
    # overview sticks (no labels) + arrows to each chain's interface region
    all_ca = np.array(list(chain_xyz.values()))
    center = all_ca.mean(0)
    for role in ('heavy', 'light', 'antigen'):
        draw(role, 0, 0, labels=False)
        ch, sel = selected(role)
        pts = np.array([chain_xyz[(ch, resi)] for v, resi, resn in sel if (ch, resi) in chain_xyz])
        if len(pts) == 0: continue
        cen = pts.mean(0)
        d = cen - center; d = d / (np.linalg.norm(d) + 1e-8)
        start = cen + d * 18.0
        view.addArrow({'start': {'x': float(start[0]), 'y': float(start[1]), 'z': float(start[2])},
                       'end':   {'x': float(cen[0]),   'y': float(cen[1]),   'z': float(cen[2])},
                       'radius': 0.5, 'radiusRatio': 3.0, 'color': ROLE[role]['arrow']}, viewer=(0, 0))
        view.addLabel(role.capitalize(),
                      {'position': {'x': float(start[0]), 'y': float(start[1]), 'z': float(start[2])},
                       'fontSize': 13, 'fontColor': 'white', 'backgroundColor': ROLE[role]['arrow'],
                       'backgroundOpacity': 0.85}, viewer=(0, 0))
    # per-chain insets with labels, zoomed to that chain
    for role in ('heavy', 'light', 'antigen'):
        gr, gc = cells[role]
        draw(role, gr, gc, labels=True)
        view.zoomTo({'chain': chains[role]}, viewer=(gr, gc))
    view.zoomTo(viewer=(0, 0))
    view.setBackgroundColor('white')
    print('Highlighted residues (>%.1f):' % ATTR_THRESHOLD,
          {r: len(selected(r)[1]) for r in ('heavy', 'light', 'antigen')})
    print('Panels:  (0,0) overview + arrows | (0,1) Heavy/green | (1,0) Light/blue | (1,1) Antigen/purple')
    view.show()
else:
    print(f'Set CHAIN_MAP[{PDB_ID!r}] = {{heavy:.., light:.., antigen:..}} for your structure.')


---
**Notes.** Zero-shot uses the model exactly as trained on SAaIntDB; few-shot starts from those
weights and adapts to your labels. Correlations (Pearson/Spearman) are the robust transfer metric;
RMSE is meaningful only when `Y` shares the SAaIntDB pKd scale. For batch reproduction of the paper
tables, see the `README`.